In [ ]:
import ast
import pandas as pd
import numpy as np
import ipynbname
from Testing.DRTLO_GD_R2 import *
from Functions.AutoCloud import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT import *
import optuna
from optuna.samplers import RandomSampler
from optuna.exceptions import TrialPruned

FileName = ipynbname.name()
out_path = f'Optimization\\eDRTLO_GD\\multi\\Optimization.csv'
RS = pd.read_excel(r'Dataset\RS.xlsx')
HI = pd.read_csv(r'Dataset\Bearing1_1.csv')
sig = HI['PC1'].values

In [ ]:
params =  [2.0, 5, '[58, 11, 64, 31, 32]', 15, 5, 0.0754386941522406,6.688979613494848e-06, 0.0029919547064531, 10.168345750070529,'ahead', 'tanh']

m,nI,nR,nO,mO,N1,N2,N3,τ,mode,act = params
X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
nI = len(Y[0])
if isinstance(nR, str): nR = ast.literal_eval(nR)

teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO+mO,ηS=[N1,N2,N3],mode=mode,act=act,
               tau=τ,rho=0.001,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
for j,_ in enumerate(X[:]):
    teda.run(X[j])
    teda.RUL_Prediction(Y[j],mode='interval',lim=len(sig)-nI+1,show=False)
    teda.WAPE_HI2(Y[j],Z[j])
    teda.Adapt(Y[j],Z[j])

#teda.c = np.append(teda.c,teda.gm)  
#PlotDSI_3D_PLT(teda)


In [ ]:
print(xxx)

In [ ]:
def Optimize(
        FileName=None,OptDim=None,OptSampler=None,OptSeed=None,OptPrune=False,
        n_study=None,timeout=None,n_trials=None,
        patience=None,
        mS =None,nIS=None,nLS=None,nRS=None,nOS =None,mOS=None,
        N1S=None,N2S=None,N3S=None,τS =None,mdS=None,actS=None):

    '''
        OptSampler modes available: None, Auto, GP, NSGAII, Random, TPE
    
        None: sampler default, suport multivariate optimization

        GP: suport multivariate optimization

        Auto: suport multivariate optimization
        
        NSGAII: poorly suport multivariate optimization
        
        Random: poorly suport multivariate optimization

        TPE: suport multivariate optimization
    '''

    if OptDim == 1 or OptDim is None: SingleObj = True
    elif OptDim > 1: SingleObj = False    
    if n_study is None: n_study = 1
    if timeout is None: timeout = 60
    if n_trials is None and patience is None: 
        n_trials = 25
        patience = 25
    elif n_trials < 100 and patience is None:
        patience = int(0.25*n_trials)
    elif n_trials >= 1e2 and n_trials < 1e3 and patience is None:
        patience = int(0.20*n_trials)
    elif n_trials >= 1e3 and n_trials < 5e3 and patience is None:
        patience = int(0.150*n_trials)
    elif n_trials >= 5e3 and n_trials < 1e4 and patience is None:
        patience = int(0.125*n_trials)    
    elif n_trials >= 1e4 and patience is None:
        patience = int(0.03*n_trials)   

    if mS  is None: mS  = [1.75,4.25]
    if nIS is None: nIS = [2,40]
    if nLS is None: nLS = [1,5]
    if nRS is None: nRS = [1,60]
    if nOS is None: nOS = [1,40]
    if mOS is None: mOS = [0,40]
    if N1S is None: N1S = [1,9]
    if N2S is None: N2S = [1,9]
    if N3S is None: N3S = [1,9]
    if τS  is None: τS  = [1,25]
    if mdS is None: mdS = [0,1]
    if actS is None: actS = [0,1,2]

    if SingleObj: names = ['MAPE_RUL*MAPE_HI', 'm', 'nI', 'nR', 'nO', 'mO', 'N1', 'N2', 'N3','TAU','past/ahead','activation']
    else: names = ['MAPE_RUL','MAPE_HI', 'm', 'nI', 'nR', 'nO', 'mO', 'N1', 'N2', 'N3','TAU','past/ahead','activation']
    study_dir, out_path = df_ParamsTable(names,FileName)

    for i in range(n_study):
        print('iteration:',i+1)
        def objective(trial):
            m = trial.suggest_float('m', mS[0], mS[1],step=0.25)
            nI = trial.suggest_int('nI', nIS[0], nIS[1]) 
            n_layers = trial.suggest_int('n_layers', nLS[0], nLS[1])
            nR = [trial.suggest_int(f'nR_layer_{l}', nRS[0], nRS[1]) for l in range(n_layers)]
            nO = trial.suggest_int('nO', nOS[0], nOS[1]) 
            N1 = trial.suggest_int('N1', N1S[0], N1S[1])
            N2 = trial.suggest_int('N2', N2S[0], N2S[1])
            N3 = trial.suggest_int('N3', N3S[0], N3S[1])
            τ = trial.suggest_float('τ', τS[0], τS[1])
            mode = trial.suggest_int('mode', mdS[0], mdS[1])  
            act = trial.suggest_int('act', actS[0], actS[1])  

            if mode == 'past' or mode == 0:
                mO = trial.suggest_int('mO', 0, 0) 
                if nO > nI: raise TrialPruned()

            elif mode == 'ahead' or mode == 1:
                mO = trial.suggest_int('mO', mOS[0], mOS[1]) 
                if   nI > 20 or nO > 20: raise TrialPruned()
                elif mO > nI: raise TrialPruned()
                
            X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
            
            teda=AutoCloud(m=m,nI=len(Y[0]),nR=nR,nO=nO+mO,ηS=[N1,N2,N3],mode=mode,act=act,
                           tau=τ,rho=0.0,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
            for j,_ in enumerate(X[:]):
                teda.run(X[j])
                teda.RUL_Prediction(Y[j],mode='single',lim=len(sig)-nI+1)
                teda.Adapt(Y[j],Z[j])
                #teda.WAPE_HI2(Y[j],Z[j])

                if np.isinf(teda.hiP[-1]) or np.isnan(teda.hiP[-1]): raise TrialPruned()
                if not OptPrune: continue
        
                if j >=60 and j%5==0:
                    if teda.wape_RUL > 0.6: raise TrialPruned()
                    if teda.wape_HI  > 0.3: raise TrialPruned()

                if SingleObj and j%35==0:
                    trial.report(teda.wape_RUL+teda.wape_HI, j)
                    if trial.should_prune():
                        raise optuna.TrialPruned()

            if SingleObj: return teda.wape_RUL + teda.wape_HI
            else: return teda.wape_RUL,teda.wape_HI

        pruner=optuna.pruners.HyperbandPruner()
        if SingleObj: study = optuna.create_study(direction='minimize',pruner=pruner,sampler=SelSampler(mode=OptSampler,seed=OptSeed))
        else: study = optuna.create_study(directions=['minimize','minimize'],sampler=SelSampler(mode=OptSampler,seed=OptSeed))
        study.optimize(objective, n_trials=n_trials, timeout=timeout, callbacks=[EarlyStoppingCallback(patience=patience)])

        vec = []
        if SingleObj:
            trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
            for trial in trials:
                p = trial.params
                nR_list = [p[f'nR_layer_{l}'] for l in range(p['n_layers'])]
                row = [trial.values[0],p['m'],p['nI'],str(nR_list),p['nO'],p['mO'],p['N1'],p['N2'],p['N3'],p['τ'],p['mode'],p['act']]   
                vec.append(row)
            df2 = pd.DataFrame(vec,columns=names)
            df2 = df2.sort_values(by=df2.columns[0], ascending=False)[-5:]

        elif not SingleObj: 
            for trial in study.best_trials:
                p = trial.params
                nR_list = [p[f'nR_layer_{l}'] for l in range(p['n_layers'])]
                row = [trial.values[0],trial.values[1],p['m'],p['nI'],str(nR_list),p['nO'],p['mO'],p['N1'],p['N2'],p['N3'],p['τ'],p['mode'],p['act']]
                vec.append(row)
            df2 = pd.DataFrame(vec,columns=names)
        
    if os.path.isfile(out_path) and out_path.startswith(study_dir):
        df1 = pd.read_csv(out_path)
        df_stdy = pd.concat([df1, df2], ignore_index=True)
    
    else: df_stdy = df2
    
    df_stdy.to_csv(out_path, index=False)
    opt_path = os.path.join(study_dir,f'opt_{len(os.listdir(study_dir))-1}.csv')
    
    if df2.shape[0] > 0: df2.to_csv(opt_path, index=False)

    return [df_stdy, df2 ]

In [ ]:
dfs = Optimize(FileName=FileName[:-4],OptDim=1,OptSampler='tpe',OptPrune=False,
                         n_study=1,timeout=1200,n_trials=1e2,patience=1e3,
                         mS=[2.0,4.5],nRS=[1,70],mdS=[1,1],actS=[0,1])

In [ ]:
for i in range(1,6):
    dfs = Optimize(FileName=FileName[:-4],OptDim=1,OptSampler='tpe',OptPrune=False,
                         n_study=6,timeout=1680,n_trials=1e4,patience=1e3,
                         mS=[2.0,4.5],nLS=[i,i],nRS=[1,70],mdS=[1,1],actS=[0,1])

c:\Users\Claudio\AppData\Local\Programs\Python\Python310\lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\Claudio\AppData\Local\Programs\Python\Python310\lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-08-13 16:31:03,919] A new study created in memory with name: no-name-66aeb12f-2231-4861-a1fd-e0c68d3a6250
[I 2026-08-13 16:31:03,921] Trial 0 pruned. 
[I 2026-08-13 16:31:03,923] Trial 1 pruned. 
[I 2026-08-13 16:31:03,925] Trial 2 pruned. 
[I 2026-08-13 16:31:03,926] Trial 3 pruned. 
[I 2026-08-13 16:31:03,928] Trial 4 pruned. 
[I 2026-08-13 16:31:03,930] Trial 5 pruned. 
[I 2026-08-13 16:31:03,957] Trial 6 finished with value: 2.882885320994888 and parameters: {'m': 2.25, 'nI': 10, 'n_layers': 1, 'nR_layer_0': 51, 'nO': 6, 'N

iteration: 1


[I 2026-08-13 16:31:04,151] Trial 11 finished with value: 0.48709536571058487 and parameters: {'m': 4.25, 'nI': 8, 'n_layers': 1, 'nR_layer_0': 63, 'nO': 14, 'N1': 2, 'N2': 8, 'N3': 9, 'τ': 19.168406714847894, 'mode': 1, 'act': 0, 'mO': 3}. Best is trial 11 with value: 0.48709536571058487.
[I 2026-08-13 16:31:04,153] Trial 12 pruned. 
[I 2026-08-13 16:31:04,156] Trial 13 pruned. 
[I 2026-08-13 16:31:04,158] Trial 14 pruned. 
[I 2026-08-13 16:31:04,160] Trial 15 pruned. 
[I 2026-08-13 16:31:04,162] Trial 16 pruned. 
[I 2026-08-13 16:31:04,164] Trial 17 pruned. 
[I 2026-08-13 16:31:04,166] Trial 18 pruned. 
[I 2026-08-13 16:31:04,167] Trial 19 pruned. 
[I 2026-08-13 16:31:04,169] Trial 20 pruned. 
[I 2026-08-13 16:31:04,170] Trial 21 pruned. 
[I 2026-08-13 16:31:04,173] Trial 22 pruned. 
[I 2026-08-13 16:31:04,174] Trial 23 pruned. 
[I 2026-08-13 16:31:04,177] Trial 24 pruned. 
[I 2026-08-13 16:31:04,179] Trial 25 pruned. 
[I 2026-08-13 16:31:04,181] Trial 26 pruned. 
[I 2026-08-13 16:31

In [ ]:
df = dfs[1]
#df = df[(df.iloc[:,0] <= 0.07)]
params_list = df.values[:,-11:]
df

In [ ]:
tedas = []
for i,params in enumerate(params_list):
    m,nI,nR,nO,mO,N1,N2,N3,τ,mode,act = params
    X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
    if isinstance(nR, str): nR = ast.literal_eval(nR)

    teda=AutoCloud(m=m,nI=len(Y[0]),nR=nR,nO=nO+mO,ηS=[N1,N2,N3],mode=mode,act=act,
                tau=τ,rho=0.001,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
    for j,_ in enumerate(X[:]):
        teda.run(X[j])
        teda.RUL_Prediction(Y[j],mode='interval',lim=len(sig)-nI+1,show=False)
        teda.Adapt(Y[j],Z[j])

    teda.c = np.append(teda.c,teda.gm)  
    tedas.append(teda)
    #PlotSeriesPLY(ySeries=[teda.wape_HI_hist,teda.wape_RUL_hist])

In [ ]:
PlotDSI_3D_PLT(tedas[0])